In [8]:
import pandas as pd
import numpy as np

# Nạp dữ liệu vào Colab
df = pd.read_csv('dulieuxettuyendaihoc.csv')
print(f"Kích thước dữ liệu gốc ban đầu: {df.shape}")

# -------------------------------------------------------------------------
# BƯỚC 1: Kiểm tra thiếu thông tin

print("\nBƯỚC 1: KIỂM TRA THIẾU THÔNG TIN")
thong_tin_thieu = df.isnull().sum()
print("Số lượng dòng bị thiếu ở mỗi cột:")
print(thong_tin_thieu[thong_tin_thieu > 0] if thong_tin_thieu.sum() > 0 else "Không có cột nào bị thiếu dữ liệu.")

# -------------------------------------------------------------------------
# BƯỚC 2: Loại bỏ hoặc điền giá trị thiếu

print("\nBƯỚC 2: XỬ LÝ ĐIỀN GIÁ TRỊ / LOẠI BỎ")
# Tự động điền các cột số bằng giá trị Trung vị và cột chữ bằng giá trị Phổ biến nhất
for cot in df.columns:
    if df[cot].isnull().sum() > 0:
        if df[cot].dtype in ['int64', 'float64']:
            df[cot] = df[cot].fillna(df[cot].median())
        else:
            df[cot] = df[cot].fillna(df[cot].mode()[0])
print("Đã hoàn thành kiểm tra và điền các giá trị thiếu (nếu có).")

# -------------------------------------------------------------------------
# BƯỚC 3: Phát hiện Outlier -> Remove (Sử dụng phương pháp IQR)

print("\nBƯỚC 3: PHÁT HIỆN VÀ LOẠI BỎ OUTLIER")
# Chọn các cột điểm số thực tế để quét Outlier (ví dụ các cột điểm toán T1, T2,..., T6 nếu có), ở đây mã sẽ quét tự động qua tất cả các cột định lượng (dạng số) trong file của bạn
cac_cot_so = df.select_dtypes(include=['int64', 'float64']).columns
dong_truoc_xoa = len(df)

for cot in cac_cot_so:
    Q1 = df[cot].quantile(0.25)
    Q3 = df[cot].quantile(0.75)
    IQR = Q3 - Q1

    # Xác định biên giới hạn cho dữ liệu hợp lệ
    bien_duoi = Q1 - 1.5 * IQR
    bien_tren = Q3 + 1.5 * IQR

    # Loại bỏ các dòng chứa giá trị nằm ngoài biên
    df = df[(df[cot] >= bien_duoi) & (df[cot] <= bien_tren)]

dong_sau_xoa = len(df)
print(f"Đã loại bỏ {dong_truoc_xoa - dong_sau_xoa} dòng dữ liệu ngoại lai.")
print(f"Kích thước dữ liệu hiện tại: {df.shape}")

# -------------------------------------------------------------------------
# BƯỚC 4: Kiểm tra tính cân bằng dữ liệu

print("\nBƯỚC 4: KIỂM TRA TÍNH CÂN BẰNG")

cot_muc_tieu = 'KT'

if cot_muc_tieu in df.columns:
    phan_phoi_lop = df[cot_muc_tieu].value_counts()
    ty_le_lop = df[cot_muc_tieu].value_counts(normalize=True) * 100

    print(f"Phân phối số lượng theo từng lớp trong cột [{cot_muc_tieu}]:")
    for bientuyetdoi, ty_le in zip(phan_phoi_lop.items(), ty_le_lop.items()):
        print(f"  - Lớp {bientuyetdoi[0]}: {bientuyetdoi[1]} dòng ({ty_le[1]:.2f}%)")

    # Đưa ra cảnh báo nhanh về tính cân bằng
    if ty_le_lop.max() / ty_le_lop.min() > 3:
        print("\n⚠️ Cảnh báo: Dữ liệu có hiện tượng BẤT CÂN BẰNG mạnh giữa các lớp!")
    else:
        print("\n✅ Đánh giá: Dữ liệu phân bố tương đối CÂN BẰNG.")
else:
    print(f"Không tìm thấy cột mục tiêu '{cot_muc_tieu}' để kiểm tra cân bằng. Hãy thay tên cột của bạn vào code.")

print("\nĐÃ HOÀN THÀNH QUY TRÌNH LÀM SẠCH DỮ LIỆU")
print(f"Kích thước file cuối cùng: {df.shape}")

Kích thước dữ liệu gốc ban đầu: (100, 56)

BƯỚC 1: KIỂM TRA THIẾU THÔNG TIN
Số lượng dòng bị thiếu ở mỗi cột:
DT    97
dtype: int64

BƯỚC 2: XỬ LÝ ĐIỀN GIÁ TRỊ / LOẠI BỎ
Đã hoàn thành kiểm tra và điền các giá trị thiếu (nếu có).

BƯỚC 3: PHÁT HIỆN VÀ LOẠI BỎ OUTLIER
Đã loại bỏ 32 dòng dữ liệu ngoại lai.
Kích thước dữ liệu hiện tại: (68, 56)

BƯỚC 4: KIỂM TRA TÍNH CÂN BẰNG
Phân phối số lượng theo từng lớp trong cột [KT]:
  - Lớp A: 38 dòng (55.88%)
  - Lớp D1: 15 dòng (22.06%)
  - Lớp B: 7 dòng (10.29%)
  - Lớp A1: 4 dòng (5.88%)
  - Lớp C: 4 dòng (5.88%)

⚠️ Cảnh báo: Dữ liệu có hiện tượng BẤT CÂN BẰNG mạnh giữa các lớp!

ĐÃ HOÀN THÀNH QUY TRÌNH LÀM SẠCH DỮ LIỆU
Kích thước file cuối cùng: (68, 56)
